# Airbnb Paris – Exp 3: Semantische Relevanz (SHAP)
- Modell: SAP ConTextTab (verarbeitet Zahlen **und** Freitexte nativ, binäre Klassifikation)
- SHAP (KernelExplainer) je Feature; Aggregation numerisch vs. Freitext

In [1]:
import numpy as np
import pandas as pd
import mlflow
import shap
import torch  # noqa: F401  (vor sap_rpt_oss laden: TORCH_LIBRARY-Doppelregistrierung vermeiden)
from sap_rpt_oss import SAP_RPT_OSS_Classifier
from sklearn.metrics import average_precision_score, roc_auc_score, classification_report, confusion_matrix

## Daten, Subsample & ConTextTab
- `cleaned_text` (numerisch + Freitext); balancierter 1:4-Kontext (120/480) zum Fitten, 30 Zeilen zum Erklären

In [2]:
LABEL = "is_top_rating"
TEXT_COLS = ["name", "description", "neighborhood_overview", "host_about"]

df = pd.read_csv("../../data/preprocessed/cleaned_text_airbnb_paris.csv", keep_default_na=False).set_index("row_id")
X = df.drop(columns=[LABEL])
y = df[LABEL].astype(int)
num_cols = [c for c in X.columns if c not in TEXT_COLS]
outlier_label = y.value_counts().idxmin()  # = 0 (rating <= 3)

rng = np.random.RandomState(42)
out_idx = y.index[y == outlier_label].to_numpy()
in_idx = y.index[y != outlier_label].to_numpy()
rng.shuffle(out_idx)
rng.shuffle(in_idx)
ctx_id = np.concatenate([out_idx[:120], in_idx[:480]])
explain_id = np.concatenate([out_idx[120:130], in_idx[480:500]])  # 10 Outlier + 20 Inlier

ctx = SAP_RPT_OSS_Classifier(max_context_size=8192, bagging=8)  # wie exp4 (gute Performance)
ctx.fit(X.loc[ctx_id], y.loc[ctx_id])  # y als Series (row_id-Index) -> korrektes Alignment im Modell
outlier_col = sorted(np.unique(y.loc[ctx_id]).tolist()).index(outlier_label)
print("Kontext:", len(ctx_id), "| erklärt:", len(explain_id), "| Features:", X.shape[1])

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Kontext: 600 | erklärt: 30 | Features: 44


## Performance-Gate (vor SHAP)
- ConTextTab auf disjunktem Hold-out (30/120, 1:4 wie exp4); AP, AUROC, Classification Report, Confusion Matrix
- SHAP-Analyse nur sinnvoll, wenn das Modell hier überzeugt

In [3]:
# Performance-Gate: SHAP nur sinnvoll, wenn ConTextTab hier überzeugt (Eval wie exp4)
test_id = np.concatenate([out_idx[130:160], in_idx[500:620]])  # 30 Outlier + 120 Inlier (1:4, disjunkt)
y_test = (y.loc[test_id] == outlier_label).astype(int)         # 1 = Outlier
proba = ctx.predict_proba(X.loc[test_id])[:, outlier_col]
pred = (ctx.predict(X.loc[test_id]) == outlier_label).astype(int)

print(f"Test: {len(test_id)} Zeilen | Outlier: {int(y_test.sum())}")
print(f"AP    = {average_precision_score(y_test, proba):.4f}  (Baseline {y_test.mean():.4f})")
print(f"AUROC = {roc_auc_score(y_test, proba):.4f}")
print(classification_report(y_test, pred, target_names=["inlier", "outlier"], digits=4, zero_division=0))

cm = confusion_matrix(y_test, pred)
display(pd.DataFrame(cm, index=["true inlier", "true outlier"], columns=["pred inlier", "pred outlier"]))

Test: 150 Zeilen | Outlier: 30
AP    = 0.6067  (Baseline 0.2000)
AUROC = 0.8829
              precision    recall  f1-score   support

      inlier     0.8626    0.9417    0.9004       120
     outlier     0.6316    0.4000    0.4898        30

    accuracy                         0.8333       150
   macro avg     0.7471    0.6708    0.6951       150
weighted avg     0.8164    0.8333    0.8183       150



,pred inlier,pred outlier
true inlier,113,7
true outlier,18,12


## SHAP (KernelExplainer, modell-agnostisch)
- Erklärt P(Outlier); je Feature ein SHAP-Wert (jede Freitextspalte = 1 Feature)

In [4]:
def predict_outlier(arr):
    d = pd.DataFrame(arr, columns=X.columns)
    d[num_cols] = d[num_cols].astype(float)
    return ctx.predict_proba(d)[:, outlier_col]

background = X.loc[ctx_id].sample(10, random_state=42)
explain = X.loc[explain_id]
sv = shap.KernelExplainer(predict_outlier, background).shap_values(explain, nsamples=100)
imp = pd.Series(np.abs(np.array(sv)).reshape(-1, X.shape[1]).mean(axis=0), index=X.columns)

  0%|          | 0/30 [00:00<?, ?it/s]

## Ergebnis: SHAP je Feature + numerisch vs. Freitext

In [5]:
table = imp.sort_values(ascending=False).round(5).to_frame("mean_abs_shap")
numeric_total = float(imp[num_cols].sum())
text_total = float(imp[TEXT_COLS].sum())
display(table)
print(f"numerisch gesamt={numeric_total:.5f}  |  freitext gesamt={text_total:.5f}")

,mean_abs_shap
host_tenure_days,0.03519
name,0.03517
estimated_occupancy_l365d,0.02758
calculated_host_listings_count,0.02651
host_is_superhost,0.02163
calculated_host_listings_count_entire_homes,0.01798
description,0.01703
minimum_nights,0.01635
instant_bookable,0.01618
accommodates,0.01468


numerisch gesamt=0.26987  |  freitext gesamt=0.06208


## Loggen

In [6]:
mlflow.set_tracking_uri("file:../../mlruns")
mlflow.set_experiment("airbnb_paris_experiment_3")
with mlflow.start_run(run_name="shap_contexttab"):
    # log every feature, prefix encodes the type (text_ vs num_)
    for c in X.columns:
        prefix = "text_" if c in TEXT_COLS else "num_"
        mlflow.log_metric(f"{prefix}{c}", float(imp[c]))
    mlflow.log_metric("numeric_total", numeric_total)
    mlflow.log_metric("text_total", text_total)

/home/debian/TFM_master_thesis/.venv/lib/python3.14/site-packages/mlflow/tracking/_tracking_service/utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)
